In [1]:
# CELL 1 — Mount Google Drive (needed every fresh runtime)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# CELL 2 — Sanity check: confirm nano-GPT folder is there
!ls /content/drive/MyDrive/

'Colab Notebooks'   palak_desai_resume.pdf
 nano-GPT	    PalakDesai_SanjivaniCollegeOfEngineeringKopargaon.mp4


In [3]:
# CELL 3 — Move into the actual repo folder (note: nano-GPT/nanoGPT, not just nano-GPT)
%cd /content/drive/MyDrive/nano-GPT/nanoGPT
!ls

/content/drive/MyDrive/nano-GPT/nanoGPT
assets		 data		 out-shakespeare-char  scaling_laws.ipynb
bench.py	 LICENSE	 __pycache__	       train.py
config		 model.py	 README.md	       transformer_sizing.ipynb
configurator.py  out-hindi-char  sample.py


In [4]:
# CELL 4 — Install dependencies (only needed once per fresh runtime)
!pip install -q torch numpy transformers datasets tiktoken tqdm flask pyngrok

In [5]:
# CELL 5 — Prepare the Shakespeare dataset (skip if data/shakespeare_char/meta.pkl already exists)
!python data/shakespeare_char/prepare.py

length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


In [6]:
# CELL 7 — Create the multilingual data folder (skip if already created)
!mkdir -p data/hindi_char

In [7]:
# CELL 8 — Write the multilingual sample text (skip if input.txt already has correct content)
%%writefile data/hindi_char/input.txt
नमस्ते दुनिया। यह एक बहुभाषी पाठ नमूना है जो चरित्र-स्तरीय भाषा मॉडल के प्रशिक्षण के लिए बनाया गया है।

Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, and other scripts to demonstrate that the tokenizer adapts to whatever characters appear in the file.

Bonjour le monde. Ceci est un exemple de texte multilingue. こんにちは世界。これは文字レベルの言語モデルを訓練するための多言語テキストサンプルです。

ज्ञान ही शक्ति है। Knowledge is power. Le savoir, c'est le pouvoir. 知識は力なり。

Overwriting data/hindi_char/input.txt


In [8]:
# CELL 9 — Copy and run the prepare script for the multilingual data (skip if data/hindi_char/meta.pkl already exists)
!cp data/shakespeare_char/prepare.py data/hindi_char/prepare.py
!python data/hindi_char/prepare.py
# Check output: vocab_size should be ~110 (not 65), train/val tokens should be in low hundreds (not ~1,000,000)

length of dataset in characters: 513
all the unique characters: 
 ',-.BCEHIKLTacdefghijklmnoprstuvwxzएकगचजञठडणतदनपबभमयरलशषसहािीुूेैॉो्।。こすたちでなにのはめりるれをんキサステデトプベモルレン世力多字文界知練言訓語識
vocab size: 111
train has 461 tokens
val has 52 tokens


In [9]:
# CELL 10 — Create and edit the multilingual training config (skip if config/train_hindi_char.py already correct)
!cp config/train_shakespeare_char.py config/train_hindi_char.py
!sed -i "s/dataset = 'shakespeare_char'/dataset = 'hindi_char'/" config/train_hindi_char.py
!sed -i "s/out_dir = 'out-shakespeare-char'/out_dir = 'out-hindi-char'/" config/train_hindi_char.py
!sed -i "s/max_iters = 5000/max_iters = 1000/" config/train_hindi_char.py
!sed -i "s/lr_decay_iters = 5000/lr_decay_iters = 1000/" config/train_hindi_char.py
!sed -i "s/block_size = 256/block_size = 16/" config/train_hindi_char.py
!sed -i "s/always_save_checkpoint = False/always_save_checkpoint = True/" config/train_hindi_char.py
!cat config/train_hindi_char.py
# Confirm: dataset, out_dir, max_iters, lr_decay_iters, block_size, always_save_checkpoint all changed

# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 16 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 1000
lr_decay_iters = 1000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 100 # not super necessary pote

In [10]:
# CELL 11 — Train the multilingual model (skip if out-hindi-char/ckpt.pt already exists from a successful run)
!python train.py config/train_hindi_char.py --compile=False

Overriding config with config/train_hindi_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-hindi-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = True

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'hindi_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 16 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 1000
lr_decay_iters = 1000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is s

In [13]:
# CELL 12 — (optional) Verify the multilingual checkpoint actually saved correctly
import torch
ckpt = torch.load('out-hindi-char/ckpt.pt', map_location='cpu')
print("saved at iter:", ckpt['iter_num'], "best val loss:", ckpt['best_val_loss'])


saved at iter: 1000 best val loss: tensor(8.0137)


In [14]:
# CELL 13 — (optional) Test generation from the multilingual model directly
!python sample.py --out_dir=out-hindi-char --num_samples=3 --max_new_tokens=200

Overriding: out_dir = out-hindi-char
Overriding: num_samples = 3
Overriding: max_new_tokens = 200
number of parameters: 10.66M
Loading meta from data/hindi_char/meta.pkl...


ज्ञान ही शक्ति है। Knowlevel language model. It mixes Hindi, English, and other scripts to demonstrate that the tokenizer adapts to whatever characters appear in the file.

Bonjour le monde. Ceci est
---------------


Hello world. This is a multilingual text sample created to train a character-level language model. It mixes Hindi, English, and other scripts to demonstrate that the tokenizer adapts to whatever char
---------------

ज्ञान ही शक्ति है। Knowlevel language model. It mixes Hindi, English, and other scripts to demonstrate that the tokenizer adapts to whatever characters appear in the file.

Bonjour le monde. Ceci est 
---------------


In [16]:
# CELL 14 — Kill any stale server process on port 3001 before restarting
!fuser -k 3001/tcp 2>/dev/null || true
import time
time.sleep(2)
!lsof -i :3001

In [17]:
# CELL 15 — Overwrite server.py with the multi-model (dropdown-capable) version
%%writefile /content/drive/MyDrive/nano-GPT/frontend/server.py
import os
import pickle
import json
from contextlib import nullcontext
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from urllib.parse import urlparse, parse_qs

import torch

FRONTEND_ROOT = os.path.dirname(os.path.abspath(__file__))
PALAKGPT_ROOT = os.path.dirname(FRONTEND_ROOT)
NANOGPT_DIR = os.path.join(PALAKGPT_ROOT, "nanoGPT")

import sys
sys.path.insert(0, NANOGPT_DIR)
from model import GPTConfig, GPT  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else "float16" if DEVICE == "cuda" else "float32"
SEED = 1337

MAX_NEW_TOKENS = 300

torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed(SEED)

device_type = "cuda" if "cuda" in DEVICE else "cpu"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[DTYPE]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

MODEL_REGISTRY = {
    "shakespeare": {
        "label": "Shakespeare (English)",
        "out_dir": os.path.join(NANOGPT_DIR, "out-shakespeare-char"),
    },
    "hindi": {
        "label": "Multilingual (Hindi/JP/FR demo)",
        "out_dir": os.path.join(NANOGPT_DIR, "out-hindi-char"),
    },
}


def load_model(ckpt_path):
    print(f"Loading checkpoint from {ckpt_path} ...")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)
    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(DEVICE)
    print("Model loaded.")
    return model, checkpoint


def load_tokenizer(checkpoint):
    meta_path = None
    if "config" in checkpoint and "dataset" in checkpoint["config"]:
        candidate = os.path.join(NANOGPT_DIR, "data", checkpoint["config"]["dataset"], "meta.pkl")
        if os.path.exists(candidate):
            meta_path = candidate

    if meta_path:
        print(f"Using char-level tokenizer from {meta_path}")
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        stoi, itos = meta["stoi"], meta["itos"]

        def encode(s):
            return [stoi[c] for c in s if c in stoi]

        def decode(tokens):
            return "".join(itos[t] for t in tokens)

        return encode, decode

    print("meta.pkl not found — falling back to GPT-2 BPE tokenizer (tiktoken)")
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
    decode = lambda tokens: enc.decode(tokens)
    return encode, decode


LOADED = {}
for key, info in MODEL_REGISTRY.items():
    ckpt_path = os.path.join(info["out_dir"], "ckpt.pt")
    if not os.path.exists(ckpt_path):
        print(f"Skipping '{key}': no checkpoint at {ckpt_path}")
        continue
    model, checkpoint = load_model(ckpt_path)
    encode, decode = load_tokenizer(checkpoint)
    LOADED[key] = {"model": model, "checkpoint": checkpoint, "encode": encode, "decode": decode}

DEFAULT_MODEL_KEY = next(iter(LOADED)) if LOADED else None


def generate_reply(prompt: str, mode: str = "creative", model_key: str = None) -> str:
    if not prompt.strip():
        return "Please enter a prompt."

    model_key = model_key if model_key in LOADED else DEFAULT_MODEL_KEY
    if model_key is None:
        return "No model loaded on the server."

    entry = LOADED[model_key]
    model_obj, encode_fn, decode_fn = entry["model"], entry["encode"], entry["decode"]

    mode_params = {
        "creative": dict(temperature=1.0, top_k=200),
        "concise": dict(temperature=0.5, top_k=50),
        "technical": dict(temperature=0.7, top_k=100),
    }
    params = mode_params.get(mode, mode_params["creative"])

    ids = encode_fn(prompt)
    if not ids:
        return "Couldn't encode that prompt with the model's tokenizer — try different text."

    x = torch.tensor(ids, dtype=torch.long, device=DEVICE)[None, ...]

    with torch.no_grad():
        with ctx:
            y = model_obj.generate(
                x,
                MAX_NEW_TOKENS,
                temperature=params["temperature"],
                top_k=params["top_k"],
            )

    full_text = decode_fn(y[0].tolist())
    completion = full_text[len(prompt):] if full_text.startswith(prompt) else full_text
    return completion.strip() or full_text.strip()


class Handler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=FRONTEND_ROOT, **kwargs)

    def do_GET(self):
        if self.path.startswith("/api/info"):
            qs = parse_qs(urlparse(self.path).query)
            requested = qs.get("model", [DEFAULT_MODEL_KEY])[0]
            model_key = requested if requested in LOADED else DEFAULT_MODEL_KEY

            entry = LOADED.get(model_key)
            if entry is None:
                info = {"models": [], "default": None, "current": None}
            else:
                model_args = entry["checkpoint"].get("model_args", {})
                dataset_name = entry["checkpoint"].get("config", {}).get("dataset", model_key)
                n_params = sum(p.numel() for p in entry["model"].parameters())
                info = {
                    "models": [{"key": k, "label": MODEL_REGISTRY[k]["label"]} for k in LOADED],
                    "default": DEFAULT_MODEL_KEY,
                    "current": {
                        "key": model_key,
                        "architecture": f"{model_args.get('n_layer', '?')} layers . "
                                         f"{model_args.get('n_head', '?')} heads . "
                                         f"{model_args.get('n_embd', '?')} dim",
                        "params": f"{n_params / 1e6:.2f}M",
                        "tokenizer": "character-level",
                        "dataset": dataset_name.replace("_", " ").title(),
                    },
                }
            body = json.dumps(info).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
            return
        super().do_GET()

    def do_POST(self):
        if self.path == "/api/generate":
            length = int(self.headers.get("Content-Length", 0))
            body = self.rfile.read(length).decode("utf-8")
            payload = json.loads(body or "{}")
            prompt = payload.get("prompt", "")
            mode = payload.get("mode", "creative")
            model_key = payload.get("model")

            try:
                reply_text = generate_reply(prompt, mode, model_key)
                status = 200
            except Exception as e:
                reply_text = f"Generation error: {e}"
                status = 500

            reply = {"reply": reply_text}
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps(reply).encode("utf-8"))
            return
        self.send_error(404)


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 3000))
    ThreadingHTTPServer.allow_reuse_address = True
    print(f"Serving frontend at http://localhost:{port}")
    ThreadingHTTPServer(("0.0.0.0", port), Handler).serve_forever()

Overwriting /content/drive/MyDrive/nano-GPT/frontend/server.py


In [18]:
# CELL 16 — Overwrite index.html with the dropdown added
%%writefile /content/drive/MyDrive/nano-GPT/frontend/index.html
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>PalakGPT - a transformer built from scratch</title>
    <link rel="preconnect" href="https://fonts.googleapis.com" />
    <link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,500;9..144,600&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet" />
    <link rel="stylesheet" href="styles.css" />
  </head>
  <body>
    <div class="app-shell">
      <aside class="sidebar">
        <div class="brand">
          <div class="brand-mark">P</div>
          <div>
            <h1>PalakGPT</h1>
            <p>a decoder-only transformer, trained from scratch</p>
          </div>
        </div>
        <div class="panel">
          <h2>Try a prompt</h2>
          <div class="chip-list" id="examplePrompts"></div>
        </div>
        <div class="panel stats-panel">
          <h2>Model</h2>
          <div class="stat-card">
            <strong>Architecture</strong>
            <span id="statArch">-</span>
          </div>
          <div class="stat-card">
            <strong>Parameters</strong>
            <span id="statParams">-</span>
          </div>
          <div class="stat-card">
            <strong>Tokenizer</strong>
            <span id="statTokenizer">-</span>
          </div>
          <div class="stat-card">
            <strong>Trained on</strong>
            <span id="statData">-</span>
          </div>
        </div>
      </aside>
      <main class="main-panel">
        <header class="topbar">
          <div>
            <p class="eyebrow">Playground</p>
            <h2>Give it an opening line</h2>
          </div>
          <button id="clearBtn" class="secondary-btn">Clear chat</button>
        </header>
        <section class="chat-window" id="chatWindow" aria-live="polite"></section>
        <section class="composer">
          <label class="visually-hidden" for="promptInput">Enter a prompt</label>
          <textarea id="promptInput" rows="4" placeholder="ROMEO:"></textarea>
          <div class="composer-actions">
            <select id="modelSelect" aria-label="Select model"></select>
            <select id="modeSelect" aria-label="Select response mode">
              <option value="creative">Creative</option>
              <option value="concise">Concise</option>
              <option value="technical">Technical</option>
            </select>
            <button id="generateBtn">Generate</button>
          </div>
        </section>
      </main>
    </div>
    <script src="app.js"></script>
  </body>
</html>

Overwriting /content/drive/MyDrive/nano-GPT/frontend/index.html


In [19]:
# CELL 17 — Overwrite app.js with model-switching logic
%%writefile /content/drive/MyDrive/nano-GPT/frontend/app.js
const examples = [
  "ROMEO:",
  "To be, or not to be,",
  "First Citizen:"
];
const chatWindow = document.getElementById('chatWindow');
const promptInput = document.getElementById('promptInput');
const generateBtn = document.getElementById('generateBtn');
const clearBtn = document.getElementById('clearBtn');
const examplePrompts = document.getElementById('examplePrompts');
const modeSelect = document.getElementById('modeSelect');
const modelSelect = document.getElementById('modelSelect');

function renderExamples() {
  examplePrompts.innerHTML = examples
    .map((example) => `<button class="chip" data-example="${example}">${example}</button>`)
    .join('');
  examplePrompts.querySelectorAll('.chip').forEach((button) => {
    button.addEventListener('click', () => {
      promptInput.value = button.dataset.example;
      promptInput.focus();
    });
  });
}

function appendMessage(text, role = 'assistant') {
  const message = document.createElement('div');
  message.className = `message ${role}`;
  message.textContent = text;
  chatWindow.appendChild(message);
  chatWindow.scrollTop = chatWindow.scrollHeight;
}

function setLoadingState(isLoading) {
  generateBtn.disabled = isLoading;
  generateBtn.textContent = isLoading ? 'Generating...' : 'Generate';
}

function createOfflineNotice() {
  return '[offline demo mode - backend unavailable. Start server.py and reload.]';
}

async function generateReply(prompt, mode, modelKey) {
  try {
    const response = await fetch('/api/generate', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ prompt, mode, model: modelKey }),
    });
    if (!response.ok) throw new Error('Backend unavailable');
    const data = await response.json();
    return data.reply || data.message || 'No reply returned.';
  } catch (error) {
    return createOfflineNotice();
  }
}

async function handleGenerate() {
  const prompt = promptInput.value.trim();
  if (!prompt) {
    promptInput.focus();
    return;
  }
  appendMessage(prompt, 'user');
  promptInput.value = '';
  setLoadingState(true);
  const mode = modeSelect.value;
  const modelKey = modelSelect.value;
  const reply = await generateReply(prompt, mode, modelKey);
  appendMessage(reply, 'assistant');
  setLoadingState(false);
}

function clearChat() {
  chatWindow.innerHTML = '';
  appendMessage('Give me an opening line and I\u2019ll continue it in the style I learned.', 'assistant');
}

async function loadModelInfo(modelKey) {
  try {
    const url = modelKey ? `/api/info?model=${encodeURIComponent(modelKey)}` : '/api/info';
    const res = await fetch(url);
    if (!res.ok) return;
    const info = await res.json();

    if (modelSelect.options.length === 0 && Array.isArray(info.models)) {
      modelSelect.innerHTML = info.models
        .map((m) => `<option value="${m.key}">${m.label}</option>`)
        .join('');
      modelSelect.value = info.default;
    }

    const c = info.current;
    if (c) {
      document.getElementById('statArch').textContent = c.architecture ?? '—';
      document.getElementById('statParams').textContent = c.params ?? '—';
      document.getElementById('statTokenizer').textContent = c.tokenizer ?? '—';
      document.getElementById('statData').textContent = c.dataset ?? '—';
    }
  } catch (e) {
    // backend not running yet - ignore
  }
}

generateBtn.addEventListener('click', handleGenerate);
clearBtn.addEventListener('click', clearChat);
modelSelect.addEventListener('change', () => loadModelInfo(modelSelect.value));
promptInput.addEventListener('keydown', (event) => {
  if (event.key === 'Enter' && !event.shiftKey) {
    event.preventDefault();
    handleGenerate();
  }
});
renderExamples();
clearChat();
loadModelInfo();

Overwriting /content/drive/MyDrive/nano-GPT/frontend/app.js


In [34]:
# CELL 18 — Start the server fresh
import subprocess, os, time
proc = subprocess.Popen(
    ["python", "/content/drive/MyDrive/nano-GPT/frontend/server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    preexec_fn=os.setpgrp,
    env={**os.environ, "PORT": "3001"},
)
time.sleep(10)
print("still running:", proc.poll())

still running: None


In [36]:
# CELL 19 — Get the public URL to open in your browser
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(3001)"))

https://3001-gpu-t4-s-kkb-usw4a1-1tlv3y6tagw8v-a.us-west4-1.prod.colab.dev


In [31]:
!lsof -i :3001

COMMAND  PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 3945 root   44u  IPv4 114969      0t0  TCP *:3001 (LISTEN)


In [33]:
!kill -9 3945
import time
time.sleep(3)
!lsof -i :3001   # should now print nothing